# RQ3 - Runtime overhead

Plots monitoring/adaptation overhead and compares wall-clock time against fixed mechanisms.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'RQ3').exists() and (ROOT / 'Paper Results').exists():
    ROOT = ROOT / 'Paper Results'

rq3 = ROOT / 'RQ3'
summary_path = rq3 / 'overhead_summary.csv'
wall_path = rq3 / 'wall_time_summary.csv'
samples_path = rq3 / 'overhead_samples.csv'

if not summary_path.exists():
    print(f'RQ3 data not found: {summary_path}')
    print('Run: python "Paper Results/run_rq3_experiments.py"')
else:
    summary = pd.read_csv(summary_path)
    wall = pd.read_csv(wall_path) if wall_path.exists() else pd.DataFrame()
    samples = pd.read_csv(samples_path) if samples_path.exists() else pd.DataFrame()

    for df in [summary, wall, samples]:
        for col in df.columns:
            if col not in ['coord_label', 'architecture', 'run_id', 'latency_source', 'performance_state', 'switch_reason']:
                df[col] = pd.to_numeric(df[col], errors='ignore')

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    da = summary[summary['coord_label'] == 'DA']
    if not da.empty and 'adapt_overhead_ms_mean_mean' in da:
        pivot = da.pivot_table(index='n_robots', columns='item_spawning_time', values='adapt_overhead_ms_mean_mean', aggfunc='mean')
        pivot.plot(marker='o', ax=ax)
        ax.set_title('Mean overhead per adaptation check')
        ax.set_xlabel('Agents')
        ax.set_ylabel('Milliseconds')
        ax.grid(alpha=0.25)
    else:
        ax.text(0.5, 0.5, 'No DA overhead summary yet', ha='center', va='center')
        ax.set_axis_off()

    ax = axes[0, 1]
    if not samples.empty:
        cols = ['total_overhead_ms', 'rtlola_query_ms', 'scoring_ms']
        means = samples[cols].apply(pd.to_numeric, errors='coerce').mean()
        means.plot(kind='bar', ax=ax, color=['#4C78A8', '#F2B701', '#54A24B'])
        ax.set_title('Overhead breakdown')
        ax.set_ylabel('Mean milliseconds per check')
        ax.grid(axis='y', alpha=0.25)
    else:
        ax.text(0.5, 0.5, 'No overhead samples yet', ha='center', va='center')
        ax.set_axis_off()

    ax = axes[1, 0]
    if not wall.empty and 'wall_time_s_mean' in wall:
        order = ['CE', 'SD', 'FD', 'DA']
        wall['coord_label'] = pd.Categorical(wall['coord_label'], categories=order, ordered=True)
        wall.sort_values('coord_label').plot(x='coord_label', y='wall_time_s_mean', kind='bar', ax=ax, legend=False, color='#72B7B2')
        ax.set_title('Wall-clock time by mechanism')
        ax.set_xlabel('Mechanism')
        ax.set_ylabel('Seconds')
        ax.grid(axis='y', alpha=0.25)
    else:
        ax.text(0.5, 0.5, 'No wall-time summary yet', ha='center', va='center')
        ax.set_axis_off()

    ax = axes[1, 1]
    if not da.empty and 'adapt_overhead_wall_pct_mean' in da:
        pivot = da.pivot_table(index='n_robots', columns='item_spawning_time', values='adapt_overhead_wall_pct_mean', aggfunc='mean')
        pivot.plot(marker='o', ax=ax)
        ax.set_title('Adaptation overhead as wall-time percentage')
        ax.set_xlabel('Agents')
        ax.set_ylabel('% wall time')
        ax.grid(alpha=0.25)
    else:
        ax.text(0.5, 0.5, 'No overhead percentage yet', ha='center', va='center')
        ax.set_axis_off()

    fig.suptitle('RQ3: Runtime overhead of monitoring and adaptation', fontsize=14)
    fig.tight_layout()
    plt.show()
    display(summary)
